In [8]:
# Initialize & Load Feature Engineered Data
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/processed/heart_disease_features.csv')

X = df.drop(columns=['HeartDiseaseorAttack'])
y = df['HeartDiseaseorAttack']

In [9]:
# Stratified Train / Validation Split

train_X, val_X, train_y, val_y = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y, # forces both sets to keep the same ~9% of positive rate
    random_state=1
)

train_y.value_counts(normalize=True)
val_y.value_counts(normalize=True)

HeartDiseaseorAttack
0.0    0.896795
1.0    0.103205
Name: proportion, dtype: float64

In [10]:
# Fit Baseline Models with Class-Imbalance handling

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(random_state=1, class_weight='balanced'),
    "LightGBM": LGBMClassifier(random_state=1, class_weight='balanced')
}

# class_weight = 'balanced' automatically adjusts weights inversely proportional to class frequencies in the input data.

for name, model in models.items():
    model.fit(train_X, train_y)

[LightGBM] [Info] Number of positive: 18974, number of negative: 164850
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015095 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 217
[LightGBM] [Info] Number of data points in the train set: 183824, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


In [11]:
# Sanity Check

for name, model in models.items():
    preds = model.predict(val_X)
    probs = model.predict_proba(val_X)[:, 1]
    print(name, preds[:10], probs[:10])

    # Confirms if every model fits and produces both a class prediction and a proba output

Logistic Regression [1. 1. 0. 1. 0. 0. 0. 0. 0. 1.] [0.70252881 0.71001281 0.09120568 0.91806623 0.16031933 0.24230455
 0.10223876 0.01872825 0.48388342 0.52995605]
Random Forest [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.] [0.49   0.2675 0.03   0.56   0.01   0.02   0.     0.01   0.12   0.13  ]
LightGBM [1. 1. 0. 1. 0. 0. 0. 0. 0. 1.] [0.67905769 0.66771192 0.08184021 0.85761778 0.10865059 0.20852472
 0.07798859 0.01867316 0.44566715 0.59746595]


In [12]:
# Save models & Split (For notebook 04_model_validation)

import joblib 

joblib.dump(models, '../data/processed/baseline_models.pkl')
joblib.dump((train_X, train_y, val_X, val_y), '../data/processed/train_val_split.pkl')

['../data/processed/train_val_split.pkl']